## Inference on test subset: coronary_instance_eomt_small_512_dinov2_skelrecall (augs)
GT vs Predictions side-by-side

In [ ]:
import yaml
import importlib
import torch
import torch.nn.functional as F
from torch.amp.autocast_mode import autocast
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches as mpatches
from pathlib import Path
from lightning import seed_everything
from PIL import Image
import warnings

seed_everything(0, verbose=False)
warnings.filterwarnings('ignore')

device = 0

CONFIG_PATH = 'configs/dinov2/coronary/instance/eomt_small_512_dinov2_skelrecall.yaml'
CKPT_PATH = 'runs/coronary_instance_eomt_small_512_dinov2_skelrecall/augs/checkpoints/best.ckpt'
TEST_IMAGES_DIR = '/home/dsa/new_seg_final/single_dataset/test/images'
TEST_LABELS_DIR = '/home/dsa/new_seg_final/single_dataset/test/labels'

CLASS_NAMES = {
    0: 'lad', 1: 'lm', 2: 'lcx', 3: 'lad_b', 4: 'lcx_b',
    5: 'inter', 6: 'rca', 7: 'pda', 8: 'pborca'
}

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

### Load dataset & model

In [ ]:
# Load data module (for img_size, num_classes, collate)
data_module_name, class_name = config['data']['class_path'].rsplit('.', 1)
data_module_cls = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config['data'].get('init_args', {})

data = data_module_cls(
    path=data_module_kwargs['path'],
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    skeleton_enabled=data_module_kwargs.get('skeleton_enabled', False),
    skeleton_num_dilations=data_module_kwargs.get('skeleton_num_dilations', 2),
    extra_augmentations_enabled=True,
)
data.setup()
print(f'img_size: {data.img_size}, num_classes: {data.num_classes}')

In [ ]:
# Build and load model from checkpoint
encoder_cfg = config['model']['init_args']['network']['init_args']['encoder']
encoder_module_name, encoder_class_name = encoder_cfg['class_path'].rsplit('.', 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=data.img_size, **encoder_cfg.get('init_args', {}))

network_cfg = config['model']['init_args']['network']
network_module_name, network_class_name = network_cfg['class_path'].rsplit('.', 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg['init_args'].items() if k != 'encoder'}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=data.num_classes,
    encoder=encoder,
    **network_kwargs,
)

lit_module_name, lit_class_name = config['model']['class_path'].rsplit('.', 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in config['model']['init_args'].items() if k != 'network'}

model = lit_cls(
    img_size=data.img_size,
    num_classes=data.num_classes,
    network=network,
    **model_kwargs,
).eval().to(device)

# Load checkpoint weights
ckpt = torch.load(CKPT_PATH, map_location=f'cuda:{device}', weights_only=False)
model.load_state_dict(ckpt['state_dict'], strict=False)
print('Checkpoint loaded successfully')

### Load test dataset

In [ ]:
from datasets.coronary_instance import CoronaryDataset

test_dataset = CoronaryDataset(
    img_dir=Path(TEST_IMAGES_DIR),
    label_dir=Path(TEST_LABELS_DIR),
    transforms=None,
)
print(f'Test dataset size: {len(test_dataset)}')

### Run inference on all test images

In [ ]:
@torch.no_grad()
def infer_instance(img):
    """Run instance segmentation inference on a single image."""
    with autocast(dtype=torch.float16, device_type='cuda'):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)

        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode='bilinear'
        )
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits, img_sizes
        )

        class_logits = class_logits_per_layer[-1]

        # Extract instances (same logic as eval_step)
        ml = mask_logits[0]  # [num_q, H, W]
        scores = class_logits[0].softmax(dim=-1)[:, :-1]  # [num_q, num_classes]
        labels = (
            torch.arange(scores.shape[-1], device=scores.device)
            .unsqueeze(0)
            .repeat(scores.shape[0], 1)
            .flatten(0, 1)
        )

        topk_scores, topk_indices = scores.flatten(0, 1).topk(
            model.eval_top_k_instances, sorted=False
        )
        labels = labels[topk_indices]
        topk_indices = topk_indices // scores.shape[-1]
        ml = ml[topk_indices]

        masks = ml > 0
        mask_scores = (
            ml.sigmoid().flatten(1) * masks.flatten(1)
        ).sum(1) / (masks.flatten(1).sum(1) + 1e-6)
        final_scores = topk_scores * mask_scores

        # Filter low-confidence predictions
        keep = final_scores > 0.3
        masks = masks[keep].cpu().numpy()
        labels = labels[keep].cpu().numpy()
        final_scores = final_scores[keep].cpu().numpy()

        # Sort by score descending
        order = np.argsort(-final_scores)
        masks = masks[order]
        labels = labels[order]
        final_scores = final_scores[order]

    return masks, labels, final_scores

In [ ]:
# Run inference on all test images
results = []
for idx in range(len(test_dataset)):
    img, target = test_dataset[idx]
    masks, labels, scores = infer_instance(img)
    results.append({
        'img': img,
        'target': target,
        'pred_masks': masks,
        'pred_labels': labels,
        'pred_scores': scores,
        'filename': test_dataset.samples[idx][0].name,
    })
    if (idx + 1) % 10 == 0:
        print(f'Processed {idx + 1}/{len(test_dataset)}')

print(f'Done. Processed {len(results)} images.')

### Visualization: GT vs Predictions

In [ ]:
from scipy import ndimage

# Color palette for classes
CLASS_COLORS = plt.cm.tab10(np.linspace(0, 1, len(CLASS_NAMES)))


def overlay_instances(ax, img_np, masks, labels, scores=None, alpha=0.45):
    """Overlay instance masks on image with class-colored semi-transparent fill."""
    ax.imshow(img_np)
    overlay = np.zeros((*img_np.shape[:2], 4), dtype=np.float32)

    legend_entries = {}
    for i in range(len(masks)):
        mask = masks[i]
        cls_id = labels[i]
        color = CLASS_COLORS[cls_id % len(CLASS_COLORS)]
        cls_name = CLASS_NAMES.get(cls_id, str(cls_id))

        # Fill mask region
        overlay[mask, :3] = color[:3]
        overlay[mask, 3] = alpha

        # Draw contour
        eroded = ndimage.binary_erosion(mask, iterations=1)
        contour = mask & ~eroded
        overlay[contour, :3] = color[:3]
        overlay[contour, 3] = 1.0

        label_str = cls_name
        if scores is not None:
            label_str += f' ({scores[i]:.2f})'
        if cls_name not in legend_entries:
            legend_entries[cls_name] = color[:3]

    ax.imshow(overlay)
    return legend_entries


def plot_gt_vs_pred(result, figsize=(18, 7)):
    """Plot image, GT, and prediction side by side."""
    img_np = result['img'].permute(1, 2, 0).cpu().numpy()
    if img_np.max() > 1:
        img_np = img_np / 255.0

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    # Original image
    axes[0].imshow(img_np)
    axes[0].set_title(f'Image: {result["filename"]}', fontsize=11)
    axes[0].axis('off')

    # Ground truth
    gt_masks = result['target']['masks'].numpy()
    gt_labels = result['target']['labels'].numpy()
    gt_legend = overlay_instances(axes[1], img_np, gt_masks, gt_labels)
    axes[1].set_title(f'Ground Truth ({len(gt_masks)} instances)', fontsize=11)
    axes[1].axis('off')

    # Predictions
    pred_legend = overlay_instances(
        axes[2], img_np, result['pred_masks'], result['pred_labels'], result['pred_scores']
    )
    axes[2].set_title(f'Prediction ({len(result["pred_masks"])} instances)', fontsize=11)
    axes[2].axis('off')

    # Combined legend
    all_legend = {**gt_legend, **pred_legend}
    if all_legend:
        legend_patches = [
            mpatches.Patch(color=c, label=n) for n, c in all_legend.items()
        ]
        fig.legend(
            handles=legend_patches, loc='lower center',
            ncol=min(len(legend_patches), 9), fontsize=9,
            bbox_to_anchor=(0.5, -0.02)
        )

    plt.tight_layout()
    plt.show()

In [ ]:
# Show all test examples
for i, result in enumerate(results):
    plot_gt_vs_pred(result)